In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

In [2]:
documents = load_faq_data()
index = build_index(documents)

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [4]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [5]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [7]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    # now messages contains the previously sent use question and response
    input=messages,
    # tools addition
    tools=[search_tool],
)

In [9]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    # if the agent wants to comment on something or it is done running (has the final message)
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment add drop join course after start FAQ"}


In [10]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            # when the agent calls specific tools
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            # if the agent wants to comment on something or it is done running (has the final message)
            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [11]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install run locally Ollama"}
function_call: search {"query":"Ollama local setup run model locally"}
function_call: search {"query":"run Ollama locally course FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install Ollama:
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. Check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content']

'To run Ollama locally:\n\n1. Install Ollama:\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. Check that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\nollama serve\n```\nOr in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you

In [12]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discover course can I still join"}
function_call: search {"query":"course enrollment late registration can I still join discovered the course"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re joining later, you can still learn from the materials and follow along, but certificate eligibility depends on the live course timeline.

If you’d like, I can also help you figure out the best way to catch up quickly. Any other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re joining later, you can still learn from the materials and follow along, but certificate eligibility depends on the live course timeline.\n\nIf you’d like, I can also help you figure out the best way to catch up quickly. Any other areas you want to explore?'

## Light-weight input guardrails

In [13]:
# restric off-topic questions and encourage multiple searches

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [14]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit chess"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry related to “queen gambit,” so it looks like this question is outside the course FAQ.

If you want, you can ask about a course topic or logistics item, and I can look it up. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry related to “queen gambit,” so it looks like this question is outside the course FAQ.\n\nIf you want, you can ask about a course topic or logistics item, and I can look it up. Are there other areas you want to explore?'